# 01 — Tratamento e Merge (fonte única de verdade)

**Tech Challenge — Fase 1 | Pós-Tech Data Analytics (FIAP)**
Base: *Brazilian E-Commerce Public Dataset by Olist* (Kaggle) — ~100 mil pedidos, 2016–2018.

Este é o **único** notebook do projeto que lê os CSVs brutos. Ele executa, nesta ordem:

1. Carga dos 8 datasets originais + inventário de qualidade
2. Conversão das colunas de data
3. Filtro do período válido (**jan/2017 a ago/2018**)
4. Propagação do filtro para as tabelas-filhas
5. Agregação de itens, pagamentos e reviews no **grão de pedido**
6. Validação cruzada (itens × pagamentos)
7. Persistência em `.parquet` na pasta `dados_tratados/`

## Decisões de tratamento (registro de governança)

| # | Decisão | Justificativa |
|---|---------|---------------|
| 1 | Recorte temporal de **2017-01-01 a 2018-08-31** | Set–dez/2016 tem volume residual (plataforma em implantação / coleta incompleta) e set–out/2018 está truncado no meio da coleta. Manter esses meses cria uma falsa impressão de "colapso do e-commerce" no gráfico da série. |
| 2 | Filtro aplicado em `order_purchase_timestamp` | É a data do **evento comercial** (a compra), não da entrega ou da aprovação. É a data que define o mês de faturamento. |
| 3 | `df_products` e `df_sellers` **não** são filtrados por data | São tabelas de **cadastro** (dimensões), não de eventos. Filtrá-las por data quebraria os joins e removeria produtos/vendedores legítimos. |
| 4 | `df_geolocation` não é persistido | A única informação exclusiva dela é lat/long, não usada em nenhuma análise. Localização de cliente vem de `df_customers` (cidade/UF). |
| 5 | Grão do `df_combined` | **1 linha = 1 pedido.** Itens, pagamentos e reviews são agregados antes do merge para não multiplicar linhas e inflar faturamento. |
| 6 | Merges com `validate=` | Todo merge declara a cardinalidade esperada. Se a premissa quebrar, o pandas levanta erro em vez de duplicar linhas silenciosamente. |

In [30]:
# %pip install -q pyarrow
try:
    import pyarrow
    print('pyarrow', pyarrow.__version__, '- OK')
except ImportError:
    print('pyarrow NAO instalado. Descomente a primeira linha desta celula e execute.')

pyarrow 16.1.0 - OK


In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DIR_BRUTO = Path('data/raw')
DIR_SAIDA = Path('data/clean')
DIR_SAIDA.mkdir(exist_ok=True)

DATA_INICIO = '2017-01-01'
DATA_FIM    = '2018-09-01'   # limite superior EXCLUSIVO (ou seja, vai ate 31/08/2018)

if not DIR_BRUTO.exists():
    raise FileNotFoundError(
        f'Pasta {DIR_BRUTO.resolve()} nao encontrada. '
        'Abra este notebook a partir da raiz do projeto (onde fica a pasta Dados/).'
    )

print('pandas       :', pd.__version__)
print('CSVs brutos  :', DIR_BRUTO.resolve())
print('Saida parquet:', DIR_SAIDA.resolve())
print('Recorte      :', DATA_INICIO, 'ate', DATA_FIM, '(exclusivo)')

pandas       : 2.2.2
CSVs brutos  : C:\Users\joao_\Desktop\João\FIAP\Entrega_final\data\raw
Saida parquet: C:\Users\joao_\Desktop\João\FIAP\Entrega_final\data\clean
Recorte      : 2017-01-01 ate 2018-09-01 (exclusivo)


## 1. Carga dos CSVs brutos

In [32]:
ARQUIVOS = {
    'df_customers':      'olist_customers_dataset.csv',
    'df_geolocation':    'olist_geolocation_dataset.csv',
    'df_order_items':    'olist_order_items_dataset.csv',
    'df_order_payments': 'olist_order_payments_dataset.csv',
    'df_order_reviews':  'olist_order_reviews_dataset.csv',
    'df_orders':         'olist_orders_dataset.csv',
    'df_products':       'olist_products_dataset.csv',
    'df_sellers':        'olist_sellers_dataset.csv',
}

brutos = {nome: pd.read_csv(DIR_BRUTO / arq) for nome, arq in ARQUIVOS.items()}

df_customers      = brutos['df_customers']
df_geolocation    = brutos['df_geolocation']
df_order_items    = brutos['df_order_items']
df_order_payments = brutos['df_order_payments']
df_order_reviews  = brutos['df_order_reviews']
df_orders         = brutos['df_orders']
df_products       = brutos['df_products']
df_sellers        = brutos['df_sellers']

for nome, df in brutos.items():
    print(f'{nome:<20} {df.shape[0]:>7,} linhas x {df.shape[1]:>2} colunas')

df_customers          99,441 linhas x  5 colunas
df_geolocation       1,000,163 linhas x  5 colunas
df_order_items       112,650 linhas x  7 colunas
df_order_payments    103,886 linhas x  5 colunas
df_order_reviews      99,224 linhas x  7 colunas
df_orders             99,441 linhas x  8 colunas
df_products           32,951 linhas x  9 colunas
df_sellers             3,095 linhas x  4 colunas


### 1.1 Inventário de qualidade dos dados brutos

In [33]:
inventario = pd.DataFrame([
    {
        'tabela':            nome,
        'linhas':            df.shape[0],
        'colunas':           df.shape[1],
        'linhas_duplicadas': int(df.duplicated().sum()),
        'colunas_com_nulos': int((df.isna().sum() > 0).sum()),
        'total_nulos':       int(df.isna().sum().sum()),
    }
    for nome, df in brutos.items()
])

display(inventario)

,tabela,linhas,colunas,linhas_duplicadas,colunas_com_nulos,total_nulos
0,df_customers,99441,5,0,0,0
1,df_geolocation,1000163,5,261831,0,0
2,df_order_items,112650,7,0,0,0
3,df_order_payments,103886,5,0,0,0
4,df_order_reviews,99224,7,0,2,145903
5,df_orders,99441,8,0,3,4908
6,df_products,32951,9,0,8,2448
7,df_sellers,3095,4,0,0,0


In [34]:
# Detalhe dos nulos por coluna
nulos = []
for nome, df in brutos.items():
    s = df.isna().sum()
    for col, qtd in s[s > 0].items():
        nulos.append({'tabela': nome, 'coluna': col, 'nulos': int(qtd),
                      'pct_nulos': round(100 * qtd / len(df), 2)})

display(pd.DataFrame(nulos).sort_values('pct_nulos', ascending=False).reset_index(drop=True))

,tabela,coluna,nulos,pct_nulos
0,df_order_reviews,review_comment_title,87656,88.34
1,df_order_reviews,review_comment_message,58247,58.70
2,df_orders,order_delivered_customer_date,2965,2.98
3,df_products,product_category_name,610,1.85
4,df_products,product_name_lenght,610,1.85
5,df_products,product_description_lenght,610,1.85
6,df_products,product_photos_qty,610,1.85
7,df_orders,order_delivered_carrier_date,1783,1.79
8,df_orders,order_approved_at,160,0.16
9,df_products,product_weight_g,2,0.01


### 1.2 Tradução e adaptação da categoria de produto

In [35]:
df_category_translation = pd.read_csv(DIR_BRUTO / 'product_category_name_translation.csv')

df_products = df_products.merge(
    df_category_translation, on='product_category_name', how='left', validate='many_to_one'
)

sem_traducao = df_products.loc[
    df_products['product_category_name'].notna() & df_products['product_category_name_english'].isna(),
    'product_category_name'
].unique()
print(f'Categorias sem tradução oficial (fallback = nome em ingles = nome original): {list(sem_traducao)}')

df_products['product_category_name_english'] = df_products['product_category_name_english'].fillna(
    df_products['product_category_name']
)

Categorias sem tradução oficial (fallback = nome em ingles = nome original): ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [36]:
NOMES_AMIGAVEIS_CATEGORIA = {
    'health_beauty':          'Saúde & Beleza',
    'watches_gifts':          'Relógios & Presentes',
    'bed_bath_table':         'Cama, Mesa & Banho',
    'sports_leisure':         'Esporte & Lazer',
    'computers_accessories':  'Computador e Acessórios',
    'furniture_decor':        'Móveis & Decoração',
    'cool_stuff':             'Hobbies',
    'housewares':             'Artigos para Casa',
    'auto':                   'Acessórios para Carro',
    'garden_tools':           'Jardinagem',
}

def _fallback_amigavel(nome_ingles):
    if pd.isna(nome_ingles):
        return nome_ingles
    return nome_ingles.replace('_', ' ').title()

df_products['product_category_name_pt'] = (
    df_products['product_category_name_english']
    .map(NOMES_AMIGAVEIS_CATEGORIA)
    .fillna(df_products['product_category_name_english'].map(_fallback_amigavel))
)

print(f"Categorias distintas (nao nulas): {df_products['product_category_name'].nunique()}")
display(df_products[['product_category_name', 'product_category_name_english', 'product_category_name_pt']].drop_duplicates().head(12))

Categorias distintas (nao nulas): 73


,product_category_name,product_category_name_english,product_category_name_pt
0,perfumaria,perfumery,Perfumery
1,artes,art,Art
2,esporte_lazer,sports_leisure,Esporte & Lazer
3,bebes,baby,Baby
4,utilidades_domesticas,housewares,Artigos para Casa
5,instrumentos_musicais,musical_instruments,Musical Instruments
6,cool_stuff,cool_stuff,Hobbies
7,moveis_decoracao,furniture_decor,Móveis & Decoração
8,eletrodomesticos,home_appliances,Home Appliances
9,brinquedos,toys,Toys


## 2. Conversão das colunas de data 

In [37]:
COLS_DATA_ORDERS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in COLS_DATA_ORDERS:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

# Datas das demais tabelas (uteis para analises de prazo e de review)
df_order_items['shipping_limit_date']    = pd.to_datetime(df_order_items['shipping_limit_date'], errors='coerce')
df_order_reviews['review_creation_date'] = pd.to_datetime(df_order_reviews['review_creation_date'], errors='coerce')
df_order_reviews['review_answer_timestamp'] = pd.to_datetime(df_order_reviews['review_answer_timestamp'], errors='coerce')

# Chave temporal padrao do projeto
df_orders['ano_mes'] = df_orders['order_purchase_timestamp'].dt.to_period('M')

display(df_orders[COLS_DATA_ORDERS].agg(['min', 'max']))

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12


## 3. Filtro do período válido

In [38]:
pedidos_mes_full = (
    df_orders.groupby('ano_mes')['order_id'].nunique()
    .rename('qtd_pedidos').reset_index()
)
pedidos_mes_full['no_recorte'] = pedidos_mes_full['ano_mes'].between(
    pd.Period('2017-01', freq='M'), pd.Period('2018-08', freq='M')
)

display(pedidos_mes_full)
print('\nPrimeira compra:', df_orders['order_purchase_timestamp'].min())
print('Ultima  compra :', df_orders['order_purchase_timestamp'].max())

,ano_mes,qtd_pedidos,no_recorte
0,2016-09,4,False
1,2016-10,324,False
2,2016-12,1,False
3,2017-01,800,True
4,2017-02,1780,True
5,2017-03,2682,True
6,2017-04,2404,True
7,2017-05,3700,True
8,2017-06,3245,True
9,2017-07,4026,True



Primeira compra: 2016-09-04 21:15:19
Ultima  compra : 2018-10-17 17:30:18


In [39]:
n_pedidos_antes = df_orders['order_id'].nunique()

df_orders = df_orders[
    (df_orders['order_purchase_timestamp'] >= DATA_INICIO) &
    (df_orders['order_purchase_timestamp'] <  DATA_FIM)
].reset_index(drop=True)

n_pedidos_depois = df_orders['order_id'].nunique()

print(f'Pedidos antes do filtro : {n_pedidos_antes:,}')
print(f'Pedidos apos o filtro   : {n_pedidos_depois:,}')
print(f'Removidos               : {n_pedidos_antes - n_pedidos_depois:,} '
      f'({100 * (n_pedidos_antes - n_pedidos_depois) / n_pedidos_antes:.2f}%)')
print(f'Meses no recorte        : {df_orders["ano_mes"].nunique()}')

Pedidos antes do filtro : 99,441
Pedidos apos o filtro   : 99,092
Removidos               : 349 (0.35%)
Meses no recorte        : 20


## 4. Propagação do filtro para as tabelas

In [40]:
linhas_antes = {nome: len(df) for nome, df in brutos.items()}

order_ids_validos    = set(df_orders['order_id'])
customer_ids_validos = set(df_orders['customer_id'])

df_order_items    = df_order_items[df_order_items['order_id'].isin(order_ids_validos)].reset_index(drop=True)
df_order_payments = df_order_payments[df_order_payments['order_id'].isin(order_ids_validos)].reset_index(drop=True)
df_order_reviews  = df_order_reviews[df_order_reviews['order_id'].isin(order_ids_validos)].reset_index(drop=True)
df_customers      = df_customers[df_customers['customer_id'].isin(customer_ids_validos)].reset_index(drop=True)

print('Filtro propagado.')

Filtro propagado.


In [41]:
depois = {
    'df_orders': df_orders, 'df_order_items': df_order_items,
    'df_order_payments': df_order_payments, 'df_order_reviews': df_order_reviews,
    'df_customers': df_customers, 'df_products': df_products, 'df_sellers': df_sellers,
}

comparativo = pd.DataFrame([
    {
        'tabela':       nome,
        'linhas_antes': linhas_antes[nome],
        'linhas_depois': len(df),
        'removidas':    linhas_antes[nome] - len(df),
        'pct_mantido':  round(100 * len(df) / linhas_antes[nome], 2),
        'filtrada_por_data': nome not in ('df_products', 'df_sellers'),
    }
    for nome, df in depois.items()
])

display(comparativo)
print(f'\nClientes unicos (customer_unique_id) no recorte: '
      f'{df_customers["customer_unique_id"].nunique():,}')

,tabela,linhas_antes,linhas_depois,removidas,pct_mantido,filtrada_por_data
0,df_orders,99441,99092,349,99.65,True
1,df_order_items,112650,112279,371,99.67,True
2,df_order_payments,103886,103520,366,99.65,True
3,df_order_reviews,99224,98879,345,99.65,True
4,df_customers,99441,99092,349,99.65,True
5,df_products,32951,32951,0,100.00,False
6,df_sellers,3095,3095,0,100.00,False



Clientes unicos (customer_unique_id) no recorte: 95,774


## 5. Agregação no grão de pedido

In [42]:
print('order_id unico em df_orders      :', df_orders['order_id'].is_unique)
print('customer_id unico em df_customers:', df_customers['customer_id'].is_unique)
print('product_id unico em df_products  :', df_products['product_id'].is_unique)
print('seller_id unico em df_sellers    :', df_sellers['seller_id'].is_unique)

print('\nPedidos com mais de 1 item      :', int((df_order_items.groupby('order_id').size() > 1).sum()))
print('Pedidos com mais de 1 pagamento :', int((df_order_payments.groupby('order_id').size() > 1).sum()))
print('Pedidos com mais de 1 review    :', int((df_order_reviews.groupby('order_id').size() > 1).sum()))

# Duplicatas completas em reviews (investigacao, convencao do projeto: keep=False)
dup_reviews = df_order_reviews[df_order_reviews.duplicated(subset=['review_id'], keep=False)]
print('Linhas com review_id repetido   :', len(dup_reviews))

order_id unico em df_orders      : True
customer_id unico em df_customers: True
product_id unico em df_products  : True
seller_id unico em df_sellers    : True

Pedidos com mais de 1 item      : 9767
Pedidos com mais de 1 pagamento : 2950
Pedidos com mais de 1 review    : 545
Linhas com review_id repetido   : 1573


In [43]:
# --- Itens: valor de produto e frete por pedido ---
itens_agg = df_order_items.groupby('order_id').agg(
    qtd_itens              = ('order_item_id', 'count'),
    qtd_produtos_distintos = ('product_id', 'nunique'),
    qtd_sellers_distintos  = ('seller_id', 'nunique'),
    valor_produtos         = ('price', 'sum'),
    valor_frete            = ('freight_value', 'sum'),
).reset_index()
itens_agg['valor_itens_total'] = itens_agg['valor_produtos'] + itens_agg['valor_frete']

# --- Pagamentos: total pago, parcelamento e meio de pagamento ---
pagamentos_agg = df_order_payments.groupby('order_id').agg(
    valor_total_pago    = ('payment_value', 'sum'),
    qtd_parcelas        = ('payment_installments', 'max'),
    qtd_meios_pagamento = ('payment_type', 'nunique'),
).reset_index()

tipo_principal = (
    df_order_payments.sort_values('payment_value', ascending=False)
    .drop_duplicates(subset='order_id')[['order_id', 'payment_type']]
    .rename(columns={'payment_type': 'tipo_pagamento'})
)
pagamentos_agg = pagamentos_agg.merge(tipo_principal, on='order_id', how='left', validate='one_to_one')

# Voucher isolado -> insumo para a hipotese de "recompra induzida por promocao"
voucher = (
    df_order_payments[df_order_payments['payment_type'] == 'voucher']
    .groupby('order_id')['payment_value'].sum().rename('valor_voucher').reset_index()
)
pagamentos_agg = pagamentos_agg.merge(voucher, on='order_id', how='left', validate='one_to_one')
pagamentos_agg['valor_voucher'] = pagamentos_agg['valor_voucher'].fillna(0.0)
pagamentos_agg['usou_voucher']  = pagamentos_agg['valor_voucher'] > 0

# --- Reviews: nota media por pedido ---
reviews_agg = df_order_reviews.groupby('order_id').agg(
    review_score_medio = ('review_score', 'mean'),
    qtd_reviews        = ('review_id', 'nunique'),
).reset_index()

for nome, df in [('itens_agg', itens_agg), ('pagamentos_agg', pagamentos_agg), ('reviews_agg', reviews_agg)]:
    print(f'{nome:<16} {len(df):>7,} linhas | order_id unico: {df["order_id"].is_unique}')

itens_agg         98,353 linhas | order_id unico: True
pagamentos_agg    99,092 linhas | order_id unico: True
reviews_agg       98,330 linhas | order_id unico: True


### 5.2 Merge principal — `df_combined`

In [44]:
df_combined = (
    df_orders
    .merge(df_customers,   on='customer_id', how='left', validate='one_to_one')
    .merge(itens_agg,      on='order_id',    how='left', validate='one_to_one')
    .merge(pagamentos_agg, on='order_id',    how='left', validate='one_to_one')
    .merge(reviews_agg,    on='order_id',    how='left', validate='one_to_one')
)

print('df_combined:', df_combined.shape)
print('1 linha por pedido:', df_combined['order_id'].is_unique)
print('Linhas == pedidos do recorte:', len(df_combined) == n_pedidos_depois)

df_combined: (99092, 27)
1 linha por pedido: True
Linhas == pedidos do recorte: True


In [45]:
# Metricas derivadas de logistica (insumo para a analise de prazo de entrega 2017 x 2018)
df_combined['prazo_entrega_dias'] = (
    df_combined['order_delivered_customer_date'] - df_combined['order_purchase_timestamp']
).dt.days

df_combined['atraso_dias'] = (
    df_combined['order_delivered_customer_date'] - df_combined['order_estimated_delivery_date']
).dt.days

# NaN onde o pedido nao foi entregue: nao afirmamos "no prazo" para quem nunca chegou
df_combined['entrega_atrasada'] = (
    df_combined['atraso_dias'].gt(0).where(df_combined['atraso_dias'].notna())
)

df_combined['ano'] = df_combined['order_purchase_timestamp'].dt.year

display(df_combined.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ano_mes,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,qtd_itens,qtd_produtos_distintos,qtd_sellers_distintos,valor_produtos,valor_frete,valor_itens_total,valor_total_pago,qtd_parcelas,qtd_meios_pagamento,tipo_pagamento,valor_voucher,usou_voucher,review_score_medio,qtd_reviews,prazo_entrega_dias,atraso_dias,entrega_atrasada,ano
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,38.71,38.71,1,2,voucher,20.59,True,4.0,1.0,8.0,-8.0,False,2017
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018-07,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,118.70,22.76,141.46,141.46,1,1,boleto,0.00,False,4.0,1.0,13.0,-6.0,False,2018
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018-08,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,179.12,179.12,3,1,credit_card,0.00,False,5.0,1.0,9.0,-18.0,False,2018
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017-11,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,72.20,72.20,1,1,credit_card,0.00,False,5.0,1.0,13.0,-13.0,False,2017
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018-02,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,19.90,8.72,28.62,28.62,1,1,credit_card,0.00,False,5.0,1.0,2.0,-10.0,False,2018


## 6. Validação cruzada: itens × pagamentos

In [46]:
TOLERANCIA = 0.01  # R$ 0,01

val = df_combined[['order_id', 'order_status', 'valor_itens_total',
                   'valor_total_pago', 'valor_voucher']].copy()
val['diferenca'] = val['valor_total_pago'] - val['valor_itens_total']

sem_itens = val['valor_itens_total'].isna()
sem_pagto = val['valor_total_pago'].isna()
comparaveis = val[~sem_itens & ~sem_pagto].copy()
bate = comparaveis['diferenca'].abs() <= TOLERANCIA

print(f'Pedidos no recorte          : {len(val):,}')
print(f'Sem itens registrados       : {int(sem_itens.sum()):,}')
print(f'Sem pagamento registrado    : {int(sem_pagto.sum()):,}')
print(f'Comparaveis                 : {len(comparaveis):,}')
print(f'  -> batem (ate R$ {TOLERANCIA:.2f})   : {int(bate.sum()):,} ({100 * bate.mean():.2f}%)')
print(f'  -> divergem               : {int((~bate).sum()):,} ({100 * (~bate).mean():.2f}%)')
print(f'\nDivergencia absoluta total  : R$ {comparaveis.loc[~bate, "diferenca"].abs().sum():,.2f}')
print(f'Faturamento total (itens)   : R$ {comparaveis["valor_itens_total"].sum():,.2f}')
print(f'Impacto relativo            : '
      f'{100 * comparaveis.loc[~bate, "diferenca"].abs().sum() / comparaveis["valor_itens_total"].sum():.3f}%')

Pedidos no recorte          : 99,092
Sem itens registrados       : 739
Sem pagamento registrado    : 0
Comparaveis                 : 98,353
  -> batem (ate R$ 0.01)   : 97,977 (99.62%)
  -> divergem               : 376 (0.38%)

Divergencia absoluta total  : R$ 3,063.19
Faturamento total (itens)   : R$ 15,786,203.57
Impacto relativo            : 0.019%


In [47]:
# De onde vem a divergencia? Status dos pedidos sem itens ou sem pagamento
print('Status dos pedidos SEM itens:')
display(val.loc[sem_itens, 'order_status'].value_counts())

print('Status dos pedidos SEM pagamento:')
display(val.loc[sem_pagto, 'order_status'].value_counts())

# Maiores divergencias, para inspecao
print('10 maiores divergencias absolutas:')
display(
    comparaveis.loc[~bate]
    .assign(diferenca_abs=lambda d: d['diferenca'].abs())
    .sort_values('diferenca_abs', ascending=False)
    .head(10)
    .drop(columns='diferenca_abs')
)

Status dos pedidos SEM itens:


order_status
unavailable    602
canceled       132
created          5
Name: count, dtype: int64

Status dos pedidos SEM pagamento:


Series([], Name: count, dtype: int64)

10 maiores divergencias absolutas:


,order_id,order_status,valor_itens_total,valor_total_pago,valor_voucher,diferenca
11748,ce6d150fb29ada17d2082f4847107665,delivered,1403.66,1586.47,0.0,182.81
70616,70b742795bc441e94a44a084b6d9ce7a,delivered,466.93,578.82,0.0,111.89
33029,996c7e73600ad3723e8627ab7bef81e4,delivered,587.90,664.43,0.0,76.53
52805,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,213.15,274.84,0.0,61.69
85132,bc2c82b0ef78d2252b6176d1972db7c9,delivered,242.01,303.02,0.0,61.01
93978,af9ffff2ce6b3defd34fd4c78857a379,delivered,413.17,466.97,0.0,53.80
68570,bfdb5bbb06458d600a33d61f5f287472,delivered,348.93,394.36,0.0,45.43
8515,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,254.45,293.89,0.0,39.44
2608,b7579d24f5b2dd3e20f2e57d0e07d170,delivered,466.28,504.37,0.0,38.09
30081,abf1130bc676c9dcadf91e24f5e30a30,delivered,459.36,496.89,0.0,37.53


Os pedidos sem itens concentram-se em `canceled`, `unavailable` e `created` — pedidos que nunca se efetivaram, portanto sem mercadoria faturada. Não é falha de dado.

In [48]:
div = comparaveis.loc[~bate].merge(
    df_combined[['order_id', 'qtd_parcelas']], on='order_id', how='left'
)

print(f'Divergencias: {len(div):,}')
print(f'  pagou MAIS que o valor dos itens : {int((div["diferenca"] > 0).sum()):,}')
print(f'  pagou MENOS                      : {int((div["diferenca"] < 0).sum()):,}')
print(f'\nDiferenca mediana: R$ {div["diferenca"].median():.2f} '
      f'({100 * (div["diferenca"] / div["valor_itens_total"]).median():.2f}% do pedido)')

print(f'\nParcelaram (qtd_parcelas > 1):')
print(f'  entre os divergentes : {100 * (div["qtd_parcelas"] > 1).mean():.1f}%')
print(f'  na base comparavel   : '
      f'{100 * (df_combined.loc[df_combined["order_id"].isin(comparaveis["order_id"]), "qtd_parcelas"] > 1).mean():.1f}%')

neg = div.loc[div['diferenca'] < 0, 'diferenca']
print(f'\nEntre os que pagaram MENOS ({len(neg)} pedidos):')
print(f'  diferenca mediana : R$ {neg.median():.2f}')
print(f'  abaixo de R$ 1,00 : {int((neg.abs() < 1).sum())} de {len(neg)} (arredondamento)')

Divergencias: 376
  pagou MAIS que o valor dos itens : 288
  pagou MENOS                      : 88

Diferenca mediana: R$ 3.06 (5.79% do pedido)

Parcelaram (qtd_parcelas > 1):
  entre os divergentes : 83.2%
  na base comparavel   : 51.5%

Entre os que pagaram MENOS (88 pedidos):
  diferenca mediana : R$ -0.01
  abaixo de R$ 1,00 : 72 de 88 (arredondamento)


**Conclusão da validação:** a base está íntegra para análise financeira.

- A grande maioria dos pedidos bate na casa do centavo.
- As divergências são majoritariamente **positivas** (o cliente pagou mais que o preço de tabela) e
  concentradas em pedidos **parcelados**, com sobrepreço mediano compatível com **juros de cartão**.
  Isso é comportamento esperado do negócio, não erro de dado.
- As poucas divergências **negativas** são quase todas de centavos (mediana de −R$ 0,01), típicas de
  arredondamento na divisão do valor em parcelas.
- O impacto total é de **~0,02% do faturamento** — irrelevante para qualquer conclusão do relatório.

> **Consequência metodológica:** para medir **faturamento**, o projeto usa `valor_produtos`
> (preço de tabela, do lado dos itens), não `valor_total_pago` — este último embute juros de
> financiamento, que são receita do emissor do cartão, não da operação.

## 7. Persistência em `.parquet`

In [49]:
TABELAS_SAIDA = {
    'df_orders':         df_orders,
    'df_order_items':    df_order_items,
    'df_order_payments': df_order_payments,
    'df_order_reviews':  df_order_reviews,
    'df_customers':      df_customers,
    'df_products':       df_products,
    'df_sellers':        df_sellers,
    'df_combined':       df_combined,
}

for nome, df in TABELAS_SAIDA.items():
    out = df.copy()
    # Parquet nao suporta PeriodDtype -> gravamos ano_mes como texto 'YYYY-MM'
    for col in out.columns:
        if isinstance(out[col].dtype, pd.PeriodDtype):
            out[col] = out[col].astype(str)
    caminho = DIR_SAIDA / f'{nome}.parquet'
    out.to_parquet(caminho, index=False)
    print(f'{nome:<20} -> {caminho}  ({caminho.stat().st_size / 1024:,.0f} KB)')

df_orders            -> data\clean\df_orders.parquet  (10,279 KB)
df_order_items       -> data\clean\df_order_items.parquet  (6,300 KB)
df_order_payments    -> data\clean\df_order_payments.parquet  (3,767 KB)
df_order_reviews     -> data\clean\df_order_reviews.parquet  (9,363 KB)
df_customers         -> data\clean\df_customers.parquet  (6,896 KB)
df_products          -> data\clean\df_products.parquet  (1,456 KB)
df_sellers           -> data\clean\df_sellers.parquet  (132 KB)
df_combined          -> data\clean\df_combined.parquet  (15,459 KB)


In [50]:
# Round-trip: reler tudo e conferir que o que foi gravado e o que esta em memoria
print('Conferencia de releitura:\n')
for nome, df in TABELAS_SAIDA.items():
    lido = pd.read_parquet(DIR_SAIDA / f'{nome}.parquet')
    ok = lido.shape == df.shape
    print(f'{nome:<20} {str(lido.shape):>16}  {"OK" if ok else "DIVERGENTE"}')

Conferencia de releitura:

df_orders                  (99092, 9)  OK
df_order_items            (112279, 7)  OK
df_order_payments         (103520, 5)  OK
df_order_reviews           (98879, 7)  OK
df_customers               (99092, 5)  OK
df_products               (32951, 11)  OK
df_sellers                  (3095, 4)  OK
df_combined               (99092, 31)  OK


### 7.1 Dicionário das colunas derivadas

As colunas originais estão documentadas no dicionário oficial do Kaggle. Abaixo, apenas as **criadas
neste notebook** — o que o avaliador precisa para reproduzir os números.

In [51]:
dicionario_derivadas = pd.DataFrame([
    ('ano_mes',                'df_orders / df_combined', 'texto AAAA-MM', 'Mes da compra (order_purchase_timestamp). Chave temporal padrao do projeto.'),
    ('ano',                    'df_combined', 'inteiro',  'Ano da compra.'),
    ('qtd_itens',              'df_combined', 'inteiro',  'Numero de linhas de item no pedido (quantidade comprada).'),
    ('qtd_produtos_distintos', 'df_combined', 'inteiro',  'Produtos diferentes (product_id) no pedido.'),
    ('qtd_sellers_distintos',  'df_combined', 'inteiro',  'Vendedores diferentes atendendo o mesmo pedido.'),
    ('valor_produtos',         'df_combined', 'R$',       'Soma de price dos itens do pedido (faturamento de mercadoria).'),
    ('valor_frete',            'df_combined', 'R$',       'Soma de freight_value dos itens do pedido.'),
    ('valor_itens_total',      'df_combined', 'R$',       'valor_produtos + valor_frete. Valor esperado do pedido.'),
    ('valor_total_pago',       'df_combined', 'R$',       'Soma de payment_value do pedido (todos os meios).'),
    ('qtd_parcelas',           'df_combined', 'inteiro',  'Maior numero de parcelas entre os pagamentos do pedido.'),
    ('qtd_meios_pagamento',    'df_combined', 'inteiro',  'Quantidade de meios de pagamento distintos usados.'),
    ('tipo_pagamento',         'df_combined', 'texto',    'Meio de pagamento que concentra o MAIOR valor do pedido.'),
    ('valor_voucher',          'df_combined', 'R$',       'Parcela do pedido paga com voucher (0 se nao houve).'),
    ('usou_voucher',           'df_combined', 'booleano', 'True se qualquer parte do pedido foi paga com voucher.'),
    ('review_score_medio',     'df_combined', 'decimal',  'Media das notas do pedido (pedidos com mais de um review).'),
    ('qtd_reviews',            'df_combined', 'inteiro',  'Quantidade de avaliacoes distintas do pedido.'),
    ('prazo_entrega_dias',     'df_combined', 'dias',     'Entrega ao cliente menos data da compra. Nulo se nao entregue.'),
    ('atraso_dias',            'df_combined', 'dias',     'Entrega real menos data estimada. Positivo = atraso, negativo = adiantado.'),
    ('entrega_atrasada',       'df_combined', 'booleano', 'True se atraso_dias > 0. Nulo (nao False) para pedidos nao entregues.'),
    ('product_category_name_english', 'df_products', 'texto', 'Traducao oficial (Kaggle) da categoria. Fallback = nome original em pt-br quando falta traducao.'),
    ('product_category_name_pt',      'df_products', 'texto', 'Nome amigavel em portugues usado no relatorio/apresentacao. Definido para as 10 categorias de maior faturamento; demais usam fallback automatico (ingles Title Case).'),
], columns=['coluna', 'tabela', 'unidade', 'definicao'])

dicionario_derivadas.to_csv(DIR_SAIDA / 'dicionario_dados_derivadas.csv', index=False, encoding='utf-8-sig')
display(dicionario_derivadas)

,coluna,tabela,unidade,definicao
0,ano_mes,df_orders / df_combined,texto AAAA-MM,Mes da compra (order_purchase_timestamp). Chav...
1,ano,df_combined,inteiro,Ano da compra.
2,qtd_itens,df_combined,inteiro,Numero de linhas de item no pedido (quantidade...
3,qtd_produtos_distintos,df_combined,inteiro,Produtos diferentes (product_id) no pedido.
4,qtd_sellers_distintos,df_combined,inteiro,Vendedores diferentes atendendo o mesmo pedido.
5,valor_produtos,df_combined,R$,Soma de price dos itens do pedido (faturamento...
6,valor_frete,df_combined,R$,Soma de freight_value dos itens do pedido.
7,valor_itens_total,df_combined,R$,valor_produtos + valor_frete. Valor esperado d...
8,valor_total_pago,df_combined,R$,Soma de payment_value do pedido (todos os meios).
9,qtd_parcelas,df_combined,inteiro,Maior numero de parcelas entre os pagamentos d...
